# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arshiyanaaz411/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Question:** Can we predict CTR for new clients better than the Week-4 baseline, using only features available before the click?

**Methods chosen:**
- Baseline: the same one from Week 4 (re-run here on the same split).
- Random Forest: handles non-linear effects and interactions, needs little tuning, and is stable on noisy data.
- Gradient Boosting (HistGradientBoosting): often stronger on tabular data and handles missing values natively.

**Why not something fancier:** the data is tabular and noisy, so a simple tree ensemble is a fair, explainable step up. A model only counts as better if it beats the baseline on the same split and the same metric, not because it is more complex.

**First look at signal:** I checked the Spearman correlation of each feature with the target (below). This is directional only. It does not prove cause.## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance

# ---- EDIT THESE 3 LINES ----
DATA_PATH = "sample_data/california_housing_train.csv"   # corrected to the available dataset
TARGET = "median_house_value"
GROUP = "housing_median_age"           # the column that must not leak across train/test
# ----------------------------

df = pd.read_csv(DATA_PATH).dropna(subset=[TARGET, GROUP]).reset_index(drop=True)
FEATURES = [c for c in df.select_dtypes("number").columns if c not in (TARGET, GROUP)]
print("rows:", len(df), "| features:", len(FEATURES), "| groups:", df[GROUP].nunique())

corr = df[FEATURES + [TARGET]].corr(method="spearman")[TARGET].drop(TARGET)
print(corr.reindex(corr.abs().sort_values(ascending=False).index).head(10))

rows: 17000 | features: 7 | groups: 52
median_income     0.680814
total_rooms       0.204217
latitude         -0.167714
households        0.108806
total_bedrooms    0.082801
longitude        -0.069728
population       -0.000781
Name: median_house_value, dtype: float64


**Split:** grouped by `GROUP` (about 80% of groups in train, 20% in test). No group appears in both sets.

**Why this is honest:** the model will be used on clients it has not seen. A random row split would put rows from the same client in both train and test, and the model could memorize client-specific patterns. That would make the score look better than it really is.

**Same split for baseline and models**, with a fixed random seed, so the comparison is fair.## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [19]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df[GROUP]))

X_train, y_train = df.loc[train_idx, FEATURES], df.loc[train_idx, TARGET]
X_test,  y_test  = df.loc[test_idx,  FEATURES], df.loc[test_idx,  TARGET]
g_train = df.loc[train_idx, GROUP]

overlap = set(df.loc[train_idx, GROUP]) & set(df.loc[test_idx, GROUP])
print("train rows:", len(train_idx), "| test rows:", len(test_idx))
print("groups in both sets (must be 0):", len(overlap))
assert len(overlap) == 0

train rows: 13969 | test rows: 3031
groups in both sets (must be 0): 0


**Same data, same split, same metric as Week 4.** Main metric: MAE (lower is better). RMSE and R² are shown as support.

**Baseline note:** ___ (write which baseline you used in Week 4 and replace the baseline line in the code with exactly that model).

**Result (measured on held-out clients):** ___ (fill after you run: which model beat the baseline, and by how much).## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [20]:
def evaluate(model, name):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    }, pred

# REPLACE this baseline with your exact Week-4 baseline model.
baseline = DummyRegressor(strategy="median")

rf = make_pipeline(SimpleImputer(strategy="median"),
                   RandomForestRegressor(n_estimators=200, min_samples_leaf=5,
                                         n_jobs=-1, random_state=42))
hgb = HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, random_state=42)

rows, preds = [], {}
for name, m in [("Baseline (Week 4)", baseline), ("Random Forest", rf), ("Gradient Boosting", hgb)]:
    r, p = evaluate(m, name)
    rows.append(r); preds[name] = p

results = pd.DataFrame(rows).set_index("model").round(4)
results["MAE vs baseline (%)"] = ((results["MAE"] - results.loc["Baseline (Week 4)", "MAE"])
                                  / results.loc["Baseline (Week 4)", "MAE"] * 100).round(1)
display(results)

# Stability check: grouped CV on the train part only (does the gain hold across folds?)
gkf = GroupKFold(n_splits=5)
for name, m in [("Baseline (Week 4)", baseline), ("Random Forest", rf), ("Gradient Boosting", hgb)]:
    maes = []
    for tr, va in gkf.split(X_train, y_train, groups=g_train):
        m.fit(X_train.iloc[tr], y_train.iloc[tr])
        maes.append(mean_absolute_error(y_train.iloc[va], m.predict(X_train.iloc[va])))
    print(f"{name}: CV MAE {np.mean(maes):.4f} +/- {np.std(maes):.4f}")

,MAE,RMSE,R2,MAE vs baseline (%)
model,,,,
Baseline (Week 4),85352.4955,113270.9681,-0.0301,0.0
Random Forest,32408.5673,49477.6866,0.8034,-62.0
Gradient Boosting,31862.0238,47832.3318,0.8163,-62.7


Baseline (Week 4): CV MAE 89657.9954 +/- 8205.8456
Random Forest: CV MAE 32776.5840 +/- 3025.1737
Gradient Boosting: CV MAE 32303.1354 +/- 2779.1140


**Where is the model wrong?** ___ (look at the error-by-bucket table below: are errors bigger for high-CTR rows? for rare groups?)

**What does it lean on?** Permutation importance (measured on the test set) shows the features whose shuffling hurts MAE the most: ___ . This shows what the model uses, not what causes CTR.

**Caveats (directional, decision-support only):** results come from one grouped split plus a CV check. Gains may not hold on a different time period or new kinds of clients.## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [21]:
best_name = results.drop("Baseline (Week 4)")["MAE"].idxmin()
best_model = {"Random Forest": rf, "Gradient Boosting": hgb}[best_name]
print("Best model on test MAE:", best_name)

# Error by target bucket
err = pd.DataFrame({"y": y_test.values, "pred": preds[best_name]})
err["abs_err"] = (err["y"] - err["pred"]).abs()
err["bucket"] = pd.qcut(err["y"], 5, duplicates="drop")
print(err.groupby("bucket", observed=True)["abs_err"].agg(["mean", "count"]).round(4))

# Worst predictions
print(err.sort_values("abs_err", ascending=False).head(10).round(4))

# Permutation importance on the test set (subsample for speed)
n = min(5000, len(X_test))
idx = np.random.RandomState(42).choice(len(X_test), n, replace=False)
pi = permutation_importance(best_model, X_test.iloc[idx], y_test.iloc[idx],
                            scoring="neg_mean_absolute_error", n_repeats=5,
                            random_state=42, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print(imp.head(10).round(5))

Best model on test MAE: Gradient Boosting
                             mean  count
bucket                                  
(22499.999, 106500.0]  21038.8040    609
(106500.0, 152000.0]   23466.7102    604
(152000.0, 201200.0]   26512.7847    606
(201200.0, 282700.0]   32715.0622    606
(282700.0, 500001.0]   55602.6310    606
             y         pred      abs_err                 bucket
1859  500001.0  122478.7754  377522.2246   (282700.0, 500001.0]
489   500001.0  182489.8730  317511.1270   (282700.0, 500001.0]
476   500001.0  196148.7065  303852.2935   (282700.0, 500001.0]
2714  489600.0  196972.6001  292627.3999   (282700.0, 500001.0]
640   500001.0  225914.7706  274086.2294   (282700.0, 500001.0]
567   500001.0  257382.3061  242618.6939   (282700.0, 500001.0]
856   476700.0  237264.7197  239435.2803   (282700.0, 500001.0]
401   500001.0  263385.9337  236615.0663   (282700.0, 500001.0]
2898  450000.0  216602.4320  233397.5680   (282700.0, 500001.0]
727   106300.0  339570.9158  23

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.